# 04 · Canales y campo receptivo
*Filtros de C_in a C_out, convoluciones 1×1, pooling, y cuánto de la imagen ve cada neurona*

Notebook del episodio 4 de **Visión: de convoluciones a ViT** ([ver la lección](https://d32vh5r3pvn9et.cloudfront.net/cursos/vision/canales-y-campo-receptivo)). Es el mismo guion del video, en texto, con cada bloque de código para correr y su salida real.

**Mecanismo:** Conv2d(3, 4, 3): peso (4, 3, 3, 3); canal 0 = suma de 3 conv de un canal + bias (dif. 1,2e-7) → gris de la flor como conv 1×1 (píxel 0,86/0,44/0,15 → 0,53); conv 1×1 3→8 = einsum → max pool 2×2 de una 4×4: [[4, 2], [2, 6]] → r_l = r_{l−1} + (k−1)·j_{l−1}, j_l = j_{l−1}·s_l: pila conv,conv,pool,conv,conv,pool,conv,conv → 3, 5, 6, 10, 14, 16, 24, 32 → gradiente de la neurona central (red al azar, 20 imágenes 64×64): zona ≠ 0 de 32×32 exacta; el centro 16×16 junta 57 %, el 8×8 18 % → ERF gaussiano (Luo et al. 2016, O(√n))

**Cómo correrlo:** con el entorno de `~/Movies/serie-vision/lab` (`cd lab && uv run --project ~/Movies/serie-alineacion/lab --group notebooks --with jupyterlab jupyter lab ../notebooks`, o en VS Code eligiendo el kernel `lab/.venv`). Todo corre offline: Gemma 2 2B sale de la caché de Hugging Face, y los entrenamientos son de juguete, con datos sintéticos.

Cada bloque de código arranca de cero (`%reset -f`): la celda de **preparación** es lo que el video no muestra, y la siguiente es exactamente el código que aparece en pantalla.

## ▶ Antes de empezar

Corré la celda de abajo una vez: baja el código de la serie y sus dependencias. Funciona en Google Colab (gratis) y en cualquier Jupyter. Para guardar tus cambios en Colab: **Archivo → Guardar una copia en Drive**.

In [ ]:
import os, subprocess, sys
SERIE = "vision"
REPO = "https://github.com/Faridmurzone/paperclase-notebooks"

def _raiz():  # carpeta del repo si ya estamos adentro
    d = os.path.abspath(os.getcwd())
    while True:
        if os.path.isfile(os.path.join(d, ".paperclase-notebooks")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

RAIZ = _raiz()
if RAIZ is None:
    RAIZ = os.path.abspath("paperclase-notebooks")
    if not os.path.isdir(RAIZ):
        subprocess.run(["git", "clone", "--depth", "1", REPO, RAIZ], check=True)
os.chdir(os.path.join(RAIZ, SERIE, "lab"))  # los bloques de código importan desde snippets/ y leen datos/
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "../requirements.txt"], check=True)
os.environ["HF_HUB_OFFLINE"] = "0"  # el notebook original corría sin red; acá baja los modelos de Hugging Face
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("listo:", os.getcwd())

In [1]:
import os
os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
import warnings; warnings.filterwarnings("ignore")

*Visión, de las convoluciones a los transformers. Episodio cuatro: canales y campo receptivo. Hasta ahora, un kernel miraba una imagen de un solo canal. Hoy vemos cómo se mezclan muchos canales, qué hace una convolución de uno por uno, qué es el pooling, y cuánto de la imagen llega a ver cada neurona cuando apilamos capas.*

### Qué vamos a ver

- Un kernel mira todos los canales de entrada a la vez
- Conv 1×1: mezclar canales en cada píxel
- Max pooling: quedarse con el máximo de cada bloque
- Campo receptivo: crece capa a capa, y se puede medir

Cuatro piezas. Primero, que un kernel no mira un canal: mira todos los canales de entrada juntos y los suma. La convolución de uno por uno, que no mira vecinos y solo mezcla canales en cada píxel. El max pooling, que achica la imagen quedándose con el máximo de cada bloque. Y el campo receptivo: la zona de la imagen original de la que depende una neurona. Tiene una fórmula, y la vamos a comprobar con gradientes.

## Parte 1 · De C_in canales a C_out

Una imagen en color ya tiene tres canales. Después de una capa convolucional tiene los que la capa quiera: dieciséis, sesenta y cuatro, quinientos doce.

### Qué guarda una nn.Conv2d

Una capa que pasa de tres canales a cuatro. La creamos con kernel de tres. Sus pesos W y su bias b. La aplicamos a una entrada al azar de tres canales de ocho por ocho. Y reconstruimos a mano el canal de salida cero: para cada uno de los tres canales de entrada, una convolución de un solo canal con su pedazo de kernel, y sumamos las tres más el bias. El peso tiene forma cuatro, tres, tres, tres: cuatro kernels, cada uno con tres rebanadas de tres por tres, una por canal de entrada. Y la reconstrucción coincide hasta la diezmillonésima.

In [2]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_pesos.py
__file__ = 'snippets/ep04_pesos.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

In [3]:
conv = nn.Conv2d(3, 4, kernel_size=3)          # 3 canales → 4 canales
W, b = conv.weight, conv.bias
print("peso:", tuple(W.shape), " bias:", tuple(b.shape))

x = torch.randn(1, 3, 8, 8)
y = conv(x)                                    # (1, 4, 6, 6)
# canal de salida 0 = suma sobre los 3 canales de entrada, + bias
a_mano = sum(F.conv2d(x[:, c:c+1], W[0:1, c:c+1]) for c in range(3)) + b[0]
print("salida:", tuple(y.shape))
print("diferencia:", f"{(y[:, 0:1] - a_mano).abs().max():.1e}")

peso: (4, 3, 3, 3)  bias: (4,)
salida: (1, 4, 6, 6)
diferencia: 1.2e-07


### Un canal de salida es una suma de canales

$$
Y_o = b_o + \sum_{c=1}^{C_{in}} W_{o,c} \star X_c
$$
<small>cada canal de entrada, con su rebanada del kernel</small>

$$
W \in \mathbb{R}^{C_{out} \times C_{in} \times k \times k} = 4 \times 3 \times 3 \times 3
$$
<small>108 pesos + 4 bias</small>

En símbolos. X sub c es el canal c de la entrada, una imagen de un solo canal; C in es cuántos hay. W sub o c es la rebanada de tres por tres del kernel o que corresponde a ese canal, y la estrellita es la correlación del episodio dos. El canal de salida o es su bias más la suma, sobre todos los canales de entrada, de cada canal filtrado con su rebanada. En el dibujo, tres mapas filtrados que se apilan y se suman en uno. Así un kernel combina color y forma: puede responder a un borde que va de rojo a verde. El tensor de pesos tiene cuatro dimensiones: C out, C in, y el alto y ancho del kernel. Acá, ciento ocho pesos más cuatro bias.

## Parte 2 · La convolución de 1 × 1

¿Tiene sentido un kernel de uno por uno? No mira ningún vecino. Pero sí mira todos los canales.

### Pasar a gris es una conv 1×1

Un ejemplo que ya conocemos: pasar una foto a gris. Los tres pesos de la suma del brillo, puestos como un kernel de un canal de salida, tres de entrada, y uno por uno de tamaño. F punto conv2d con ese kernel calcula, en cada píxel, la suma pesada de sus tres canales, sin mirar a los vecinos. Y en general, con ocho canales de salida, es una multiplicación de matrices por el vector de canales de cada píxel; einsum la escribe con índices. Coincide exacto con la suma del episodio dos, y en un píxel del centro de la flor: rojo cero coma ochenta y seis, verde cero coma cuarenta y cuatro, azul cero coma quince dan cero coma cincuenta y tres.

**Cada línea, su término:**

- `w = torch.tensor([0.299` → $W \in \mathbb{R}^{1 \times 3 \times 1 \times 1}$ — un canal de salida, tres de entrada
- `gris = F.conv2d(x, w)` → $g_{ij} = \sum_{c} w_c\, x_{c,ij}$ — en cada píxel, por separado
- `torch.einsum` → $y_{:,ij} = W\, x_{:,ij}$ — matriz de C_out × C_in por el vector de canales

In [4]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_1x1.py
__file__ = 'snippets/ep04_1x1.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import numpy as np
import torch
import torch.nn.functional as F
from _cifar import ruta_lab

foto = torch.from_numpy(np.load(ruta_lab("datos", "fotos.npz"))["flor"]).float() / 255

In [5]:
x = foto.permute(2, 0, 1)[None]                 # (1, 3, H, W)
w = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)
gris = F.conv2d(x, w)                           # conv 1×1: 3 → 1 canal
print("gris:", tuple(gris.shape))
print("igual a foto @ pesos:",
      torch.allclose(gris[0, 0], foto @ w.flatten()))
r, g, b = x[0, :, 210, 250].tolist()            # un píxel de la flor
print(f"R {r:.2f} G {g:.2f} B {b:.2f} → gris {gris[0, 0, 210, 250]:.2f}")
w8 = torch.randn(8, 3, 1, 1)                    # 3 → 8 canales
y = torch.einsum("oc,nchw->nohw", w8[:, :, 0, 0], x)
print("conv 1×1 = matmul:", torch.allclose(F.conv2d(x, w8), y, atol=1e-5))

gris: (1, 1, 427, 640)
igual a foto @ pesos: True
R 0.86 G 0.44 B 0.15 → gris 0.53
conv 1×1 = matmul: True


In [6]:
# Verificación extra (corrió junto con el video)
json.dump({"rgb": [round(v, 2) for v in (r, g, b)], "gris": round(float(gris[0, 0, 210, 250]), 2)},
          open(ruta_lab("datos", "figs", "ep04_1x1.json"), "w"))

### Una neurona por píxel

$$
g = 0{,}299\,R + 0{,}587\,G + 0{,}114\,B
$$
<small>el píxel (210, 250) de la flor</small>

$$
y_{:,ij} = W\, x_{:,ij},\quad W \in \mathbb{R}^{C_{out}\times C_{in}}
$$
<small>lo mismo con muchas salidas</small>

Mirado de cerca, es una neurona de toda la vida. En el dibujo, el píxel real de la flor: sus tres canales entran con los pesos cero coma doscientos noventa y nueve, cero coma quinientos ochenta y siete y cero coma ciento catorce; se multiplican, se suman, y sale cero coma cincuenta y tres. Sin bias y sin activación. Con C out salidas, W es una matriz de C out por C in, y x sub dos puntos i j es el vector con los canales del píxel i j: los dos puntos quieren decir todos los canales. En el dibujo, una fila de W por ese vector da un canal de salida. Por eso las convoluciones de uno por uno se usan para cambiar la cantidad de canales barato, sin tocar el tamaño.

## Parte 3 · Max pooling

La otra pieza que se apila entre convoluciones no tiene pesos: achica.

### Una entrada de 4 × 4

Una entrada chiquita de cuatro por cuatro. Partila en cuatro bloques de dos por dos que no se pisan. El de arriba a la izquierda tiene uno, tres, cuatro y dos: su máximo es cuatro. Arriba a la derecha, el máximo es dos. Abajo a la izquierda, dos. Abajo a la derecha, seis. El max pooling devuelve exactamente esos cuatro números.

### F.max_pool2d

En código. La misma matriz. F punto max pool dos d con kernel de dos; si no le decís el stride, usa el mismo valor que el kernel, así que salta de a dos y los bloques no se pisan. Imprimimos las formas. Sale cuatro, dos, dos, seis, y la imagen pasó de cuatro por cuatro a dos por dos. No hay nada que aprender: es una regla fija. Lo que gana es lo que medimos en el episodio anterior, tolerancia a corrimientos chicos, y la mitad de resolución en cada lado.

In [7]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_pool.py
__file__ = 'snippets/ep04_pool.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import torch
import torch.nn.functional as F

In [8]:
x = torch.tensor([[1., 3., 0., 2.],
                  [4., 2., 1., 1.],
                  [0., 1., 5., 0.],
                  [2., 0., 3., 6.]])
y = F.max_pool2d(x[None, None], kernel_size=2)   # stride = 2 por defecto
print(y[0, 0])
print("forma:", tuple(x.shape), "→", tuple(y.shape[-2:]))

tensor([[4., 2.],
        [2., 6.]])
forma: (4, 4) → (2, 2)


## Parte 4 · Campo receptivo

Cada número de salida depende de una zona de la imagen original. ¿De qué tamaño es esa zona después de varias capas?

### La cuenta, capa por capa

Una pila típica: dos convoluciones de tres, un pooling, dos convoluciones más, otro pooling, y dos más. Cada capa con su kernel k y su stride s. Arrancamos con r igual a uno, un píxel ve un píxel, y j igual a uno: entre dos vecinos hay un píxel de distancia. Cada capa agrega k menos uno, multiplicado por el salto actual. Y cada stride multiplica el salto. Mirá cómo crece: tres, cinco, seis con el primer pooling, y después de a cuatro, porque los vecinos ya están a dos píxeles. Al final, una neurona de la última capa depende de un cuadrado de treinta y dos por treinta y dos.

In [9]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_campo.py
__file__ = 'snippets/ep04_campo.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)


In [10]:
capas = [("conv1", 3, 1), ("conv2", 3, 1), ("pool1", 2, 2),
         ("conv3", 3, 1), ("conv4", 3, 1), ("pool2", 2, 2),
         ("conv5", 3, 1), ("conv6", 3, 1)]       # (nombre, k, stride)
r, j = 1, 1                       # un píxel ve 1 píxel; salto 1
for nombre, k, s in capas:
    r = r + (k - 1) * j           # cuánto agrega este kernel
    j = j * s                     # distancia entre vecinos, en píxeles
    print(f"{nombre}: campo {r}×{r}, salto {j}")

conv1: campo 3×3, salto 1
conv2: campo 5×5, salto 1
pool1: campo 6×6, salto 2
conv3: campo 10×10, salto 2
conv4: campo 14×14, salto 2
pool2: campo 16×16, salto 4
conv5: campo 24×24, salto 4
conv6: campo 32×32, salto 4


### La fórmula del campo receptivo

$$
r_l = r_{l-1} + (k_l - 1)\, j_{l-1}
$$
<small>r: lado del campo; k: kernel de la capa l</small>

$$
j_l = j_{l-1}\, s_l = \prod_{m \le l} s_m
$$
<small>j: salto en píxeles; s: stride</small>

En símbolos, con l numerando las capas. r sub l es el lado del campo receptivo después de la capa l. Cada capa le suma k sub l menos uno, por j sub l menos uno, el salto que había antes. En el dibujo, el aporte de cada capa: uno de arranque, dos y dos de las primeras convoluciones, uno del pooling, cuatro y cuatro, dos, ocho y ocho. Suman treinta y dos. Y el salto j sub l es el producto de todos los strides hasta ahí; la pi grande es eso, un producto. Por eso el pooling es tan efectivo para ver lejos: no agrega mucho en el momento, pero duplica lo que agregan todas las capas que vienen después.

### El campo crece más rápido después de cada pooling

Graficado. Al principio crece de a dos por capa. Después del primer pooling, de a cuatro, y después del segundo, de a ocho. Sin pooling ni stride, ocho capas de tres por tres verían apenas diecisiete píxeles. Con dos poolings, treinta y dos.

## Parte 5 · Medirlo con el gradiente

La fórmula dice de qué píxeles puede depender una neurona. El gradiente dice de cuáles depende de verdad.

### El gradiente de una neurona respecto de la imagen

Armamos la misma pila de la tabla. Con pesos al azar, sin entrenar, y activación ReLU, rectified linear unit, que deja pasar los positivos y pone en cero los negativos. Una imagen al azar de sesenta y cuatro por sesenta y cuatro que guarda su gradiente. Elegimos la neurona del centro del mapa de salida y hacemos backward: el gradiente dice cuánto cambiaría esa neurona si movemos cada píxel. Acumulamos su valor absoluto sobre veinte imágenes. La zona donde el gradiente no es cero. Y qué parte del total cae en cuadrados centrales cada vez más grandes. La zona mide treinta y dos por treinta y dos: la fórmula exacta. Pero el cuadrado del medio, de dieciséis, que es un cuarto del área, ya junta el cincuenta y siete por ciento.

In [11]:
%reset -f
import torch; torch.set_grad_enabled(True)  # %reset no borra el modo sin gradientes de un bloque anterior
# Preparación (no aparece en el video). Snippet: lab/snippets/ep04_gradiente.py
__file__ = 'snippets/ep04_gradiente.py'  # algunos snippets importan módulos hermanos (lab/snippets/_*.py, lab/comun)
import json
import os
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import torch
import torch.nn as nn
from _cifar import ruta_lab

torch.manual_seed(0)

In [12]:
conv = lambda cin, cout: [nn.Conv2d(cin, cout, 3, padding=1), nn.ReLU()]
red = nn.Sequential(*conv(3, 16), *conv(16, 16), nn.MaxPool2d(2),
                    *conv(16, 32), *conv(32, 32), nn.MaxPool2d(2),
                    *conv(32, 32), *conv(32, 32))          # pesos al azar
G = torch.zeros(64, 64)
for _ in range(20):                                 # 20 imágenes al azar
    x = torch.randn(1, 3, 64, 64, requires_grad=True)
    y = red(x)                                      # (1, 32, 16, 16)
    y[0, :, 8, 8].sum().backward()                  # una posición del centro
    G += x.grad[0].abs().sum(0)
filas, cols = torch.nonzero(G).T
print("zona con gradiente ≠ 0:",
      f"{filas.max() - filas.min() + 1}×{cols.max() - cols.min() + 1}")
f0, c0 = filas.min(), cols.min()                     # esquina de la zona
for lado in (8, 16, 32):
    m = (32 - lado) // 2
    c = G[f0 + m:f0 + m + lado, c0 + m:c0 + m + lado]
    print(f"centro {lado:2d}×{lado:<2d}: {c.sum() / G.sum():.0%} del gradiente")

zona con gradiente ≠ 0: 32×32
centro  8×8 : 18% del gradiente
centro 16×16: 57% del gradiente
centro 32×32: 100% del gradiente


In [13]:
# Verificación extra (corrió junto con el video)
f0, c0 = int(f0), int(c0)
rec = G[f0:f0 + 32, c0:c0 + 32]
json.dump({"G": [[round(float(v), 3) for v in f] for f in rec / rec.max()], "origen": [f0, c0]},
          open(ruta_lab("datos", "figs", "ep04_gradiente.json"), "w"))

### Teórico contra efectivo

$$
G_{ij} = \sum_{n=1}^{20} \sum_{c} \left| \frac{\partial y_{\text{centro}}}{\partial x^{(n)}_{c,ij}} \right|
$$
<small>cuánto pesa cada píxel, acumulado en 20 imágenes</small>

$$
G_{ij} \neq 0 \iff (i, j) \in \text{campo de } r_L \times r_L
$$
<small>el borde del campo existe, pero casi no pesa</small>

Lo que medimos. La d curva es una derivada parcial: cuánto cambia la neurona del centro si tocamos solo el píxel i j del canal c de la imagen n. G suma su valor absoluto sobre los canales y sobre las veinte imágenes. En el dibujo, qué parte del total cae en cada cuadrado central: dieciocho por ciento en el de ocho, cincuenta y siete en el de dieciséis, todo en el de treinta y dos. La flecha doble se lee si y solo si: G es distinto de cero exactamente dentro del campo teórico de la última capa, de r sub L por r sub L. Pero no pesa parejo: el centro manda y los bordes casi no cuentan. A esa zona que de verdad pesa se le dice campo receptivo efectivo.

### El campo receptivo efectivo

Así se ve G dentro del campo teórico de treinta y dos por treinta y dos. Una mancha que brilla en el centro y se apaga hacia los bordes, con forma de campana. Los píxeles del centro llegan a la neurona por muchísimos caminos a través de las capas; los del borde, por muy pocos. Por eso pesan menos.

> **Luo et al., 2016 — Understanding the Effective Receptive Field in Deep Convolutional Neural Networks** · [arXiv:1701.04128](https://arxiv.org/abs/1701.04128)
>
> **O(√n)** · El campo receptivo efectivo «has a Gaussian distribution and only occupies a fraction of the full theoretical receptive field»; su radio crece como √n con n capas.

Esto lo estudiaron Luo y coautores. Muestran que el campo receptivo efectivo tiene forma de campana gaussiana y ocupa solo una fracción del teórico. Y algo más fuerte: con n capas, el teórico crece proporcional a n, pero el radio del efectivo crece como raíz de n. Apilar capas agranda lo que la red ve, pero más despacio de lo que dice la fórmula.

### Qué nos llevamos para diseñar redes

- Varias 3×3 apiladas en vez de un kernel grande
- Pooling o stride para ver lejos sin muchas capas
- Conv 1×1 para cambiar canales barato
- El campo efectivo es más chico: medilo

Cuatro reglas que salen de todo esto. Dos convoluciones de tres por tres apiladas ven cinco por cinco, con menos pesos que una de cinco por cinco y una no linealidad más en el medio. Para que la red vea objetos grandes, hace falta pooling o stride, que multiplican el salto. Para cambiar la cantidad de canales, las de uno por uno son lo más barato. Y no confíes en el número teórico: el gradiente te dice qué mira la red de verdad.

## Resumiendo

- Peso (C_out, C_in, k, k): cada salida suma todos los canales
- Conv 1×1 = matmul por píxel (gris = 0,299R + 0,587G + 0,114B)
- Max pooling: máximo de cada bloque, sin pesos
- Campo: r += (k − 1)·salto; el pooling multiplica el salto
- Gradiente ≠ 0 en 32 × 32, pero el centro 16 × 16 junta el 57 %
- Próximo: apilar todo en una red y entrenarla (de LeNet a AlexNet)